# Experiment 4: Handling Class Imbalance
Datasets often have many more "neutral" or "positive" comments than "negative" ones. This imbalance can lead to a model that predicts the majority class well but fails on the minority class.

### Techniques to Handle Imbalance:
1. **Class Weights**: penalizes mistakes in the minority class more heavily during training.
2. **SMOTE (Synthetic Minority Over-sampling Technique)**: Creates synthetic examples of the minority class using K-Nearest Neighbors.
3. **ADASYN**: Similar to SMOTE but focuses on examples in the minority class that are "harder" to learn.
4. **Random Undersampling**: Randomly removes samples from the majority class to balance the dataset.

### Why use them?
- **Accuracy Trap**: In an imbalanced dataset, a model that always predicts the majority class might have 90% accuracy but 0% recall for the minority class.
- **Fairness/Reliability**: We want the model to be equally good at detecting all sentiment types.

In [1]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline


c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Imbalance Learning Exploration - 2")

2026/04/14 11:49:55 INFO mlflow.tracking.fluent: Experiment with name 'Imbalance Learning Exploration - 2' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/15', creation_time=1776147595405, experiment_id='15', last_update_time=1776147595405, lifecycle_stage='active', name='Imbalance Learning Exploration - 2', tags={}, workspace='default'>

In [3]:
# Load and preprocess
from sklearn.preprocessing import RobustScaler


data = pd.read_csv(r'../data/processed/dataset.csv')
data = data.dropna(subset=['text_processed', 'sentiment']).drop_duplicates()

X_text = data['text_processed']
X_numeric = data[['char_count', 'word_count', 'avg_word_len']].values
y = data['sentiment']

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_numeric, y, test_size=0.2, random_state=42, stratify=y
)

# Text + Scaling setup
tfidf = TfidfVectorizer(max_features=2500, ngram_range=(1,1))
scaler = RobustScaler()

X_text_train_vec = tfidf.fit_transform(X_text_train)
X_text_test_vec = tfidf.transform(X_text_test)
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)

X_train_final = hstack([X_text_train_vec, X_num_train_scaled])
X_test_final = hstack([X_text_test_vec, X_num_test_scaled])

In [4]:
methods = [
    ("Baseline_NoWeights", LogisticRegression(max_iter=1000, random_state=42)),
    ("ClassWeights_Balanced", LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')),
    ("SMOTE", Pipeline([('smote', SMOTE(random_state=42)), ('clf', LogisticRegression(max_iter=1000, random_state=42))])),
    ("ADASYN", Pipeline([('adasyn', ADASYN(random_state=42)), ('clf', LogisticRegression(max_iter=1000, random_state=42))])),
    ("RandomUnderSampler", Pipeline([('rus', RandomUnderSampler(random_state=42)), ('clf', LogisticRegression(max_iter=1000, random_state=42))]))
]

for name, model in methods:
    with mlflow.start_run(run_name=f"Imbalance_{name}"):
        # Fit model (Pipeline handles the resampling internally for train set only)
        model.fit(X_train_final, y_train)
        y_pred = model.predict(X_test_final)
        
        # ── Log Parameters ────────────────────────────────────────────────────────
        mlflow.log_params({
                   # Split
            "test_size"             : 0.2,
            "stratify"              : True,
            "random_state"          : 42,

            # Feature Representation
            "representation"        : "TFIDF_2500",
            "vectorizer_type"       : "TfidfVectorizer",
            "tfidf_max_features"    : 2500,
            "ngram_range"           : "(1,1)",

            # Scaling
            "scaler"                : "RobustScalar",
            "scaled_features"       : "char_count, word_count, avg_word_len",

            # Imbalance Method
            "imbalance_method"      : name,

            # Model
            "model"                 : "LogisticRegression",
            "max_iter"              : 1000,
            "solver"                : "lbfgs",
            "random_state"          : 42,
            "class_weight"          : "balanced" if "ClassWeights" in name else "None",

            # Features
            "num_custom_features"   : 3,
            "text_features_count"   : X_text_train_vec.shape[1],
        })

        # Metrics Calculation
        report_dict:dict = classification_report(y_test, y_pred, output_dict=True) # type: ignore
        
        metrics = {
            "accuracy"         : accuracy_score(y_test, y_pred),
            "f1_weighted"      : report_dict['weighted avg']['f1-score'],
            "f1_macro"         : report_dict['macro avg']['f1-score'],
            "precision_weighted": report_dict['weighted avg']['precision'],
            "precision_macro"  : report_dict['macro avg']['precision'],
            "recall_weighted"  : report_dict['weighted avg']['recall'],
            "recall_macro"     : report_dict['macro avg']['recall'],
        }
        
        # Log Per-Class Metrics
        # Get classes from the last step of pipeline or from model
        classes = model.classes_ if hasattr(model, 'classes_') else model.steps[-1][1].classes_
        for label in classes:
            metrics[f"f1_class_{label}"] = report_dict[str(label)]['f1-score']
            metrics[f"precision_class_{label}"] = report_dict[str(label)]['precision']
            metrics[f"recall_class_{label}"] = report_dict[str(label)]['recall']
            
        mlflow.log_metrics(metrics)
        
        print(f"Method: {name} -> F1 (Weighted): {metrics['f1_weighted']:.4f}")


Method: Baseline_NoWeights -> F1 (Weighted): 0.8866
🏃 View run Imbalance_Baseline_NoWeights at: http://localhost:5000/#/experiments/15/runs/62d6577eb4ed4f5087fc89a3ef96e5dc
🧪 View experiment at: http://localhost:5000/#/experiments/15
Method: ClassWeights_Balanced -> F1 (Weighted): 0.8897
🏃 View run Imbalance_ClassWeights_Balanced at: http://localhost:5000/#/experiments/15/runs/cd02d801a83446f282bc22887640860c
🧪 View experiment at: http://localhost:5000/#/experiments/15
Method: SMOTE -> F1 (Weighted): 0.8884
🏃 View run Imbalance_SMOTE at: http://localhost:5000/#/experiments/15/runs/8ce2b2da67ef492c9cd768aea929ac64
🧪 View experiment at: http://localhost:5000/#/experiments/15
Method: ADASYN -> F1 (Weighted): 0.8825
🏃 View run Imbalance_ADASYN at: http://localhost:5000/#/experiments/15/runs/e272274599b54789a2c46aee5911b380
🧪 View experiment at: http://localhost:5000/#/experiments/15
Method: RandomUnderSampler -> F1 (Weighted): 0.8748
🏃 View run Imbalance_RandomUnderSampler at: http://local

#### Conclusion
class weights = Balanced is performing well at least for logistic regression as basemodel. 
but i will have to assess this again while checking for other models